# การจำแนกเพศสภาพจากภาพใบหน้า (Gender Classification Model)
**Dataset: SCUT-FBP5500 Facial Dataset (via kagglehub)**

* **X (Features)**: พิกเซลและพิกัดโครงสร้างใบหน้า 224×224
* **y (Target/Label)**: เพศสภาพ (0 = ชาย, 1 = หญิง)
* **Data Splitting**: Train 70% / Val 15% / Test 15%

In [ ]:
# 1. ติดตั้ง kagglehub และนำเข้าไลบรารีสำหรับทำนายเพศ (Classification)
!pip install -q kagglehub

import os
import cv2
import kagglehub
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded successfully!')

In [ ]:
# 2. Data Preparation: โหลดข้อมูลภาพ SCUT-FBP5500 สุ่ม 589 ตัวอย่าง (ชาย 295 / หญิง 294)
path = kagglehub.dataset_download('pranavchandane/scut-fbp5500-v2-facial-beauty-scores')
labels_file = os.path.join(path, 'labels.txt')
df = pd.read_csv(labels_file, sep=' ', header=None, names=['image_name', 'score'])

# โฟลเดอร์รูปภาพใบหน้า
image_dir = os.path.join(path, 'Images', 'Images')
if not os.path.exists(image_dir):
    image_dir = os.path.join(path, 'Images')

# สุ่ม 589 ตัวอย่างตามสไลด์
male_df = df[df['image_name'].str[1] == 'M'].sample(n=295, random_state=42)
female_df = df[df['image_name'].str[1] == 'F'].sample(n=294, random_state=42)
sample_df = pd.concat([male_df, female_df]).reset_index(drop=True)

# X (Features): พิกเซลภาพใบหน้าย่อขนาด 28x28 (784 features)
# y (Target): 0 = male, 1 = female
X = []
y = []
valid_img_paths = []

for _, row in sample_df.iterrows():
    img_path = os.path.join(image_dir, row['image_name'])
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        feat = cv2.resize(img, (28, 28)).flatten() / 255.0
        X.append(feat)
        y.append(1 if row['image_name'][1] == 'F' else 0)
        valid_img_paths.append(img_path)

X = np.array(X)
y = np.array(y)

print(f'Total face samples: {len(X)}')
print(f'Male samples: {np.sum(y == 0)}, Female samples: {np.sum(y == 1)}')


In [ ]:
# 3. Data Splitting: Train 70% / Val 15% / Test 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Training Set   (70%): {len(X_train)} samples')
print(f'Validation Set (15%): {len(X_val)} samples')
print(f'Test Set       (15%): {len(X_test)} samples')

In [ ]:
# 4. สร้างและฝึกฝนโมเดลจำแนกเพศ (Logistic Regression)
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_acc = accuracy_score(y_val, model.predict(X_val))
print(f'Validation Accuracy: {val_acc:.4f}')

In [ ]:
# 5. ทดสอบประสิทธิภาพขั้นสุดท้ายด้วย Test Set
answer = model.predict(X_test)

print(f'Score: {accuracy_score(y_test, answer):.4f}\n')
print(classification_report(y_test, answer, target_names=['male', 'female']))
print(confusion_matrix(y_test, answer))

In [ ]:
# 6. Inference Demo: แสดงผลลัพธ์การทำนายเพศจากภาพใบหน้าจริง
# ทดสอบกับรูปแรกในชุดข้อมูล
sample_img_path = valid_img_paths[0]
display_img = Image.open(sample_img_path)
display(display_img.resize((160, 160)))

# สกัด Feature จากภาพที่นำมาแสดงโดยตรง
test_img = cv2.imread(sample_img_path, cv2.IMREAD_GRAYSCALE)
test_feat = cv2.resize(test_img, (28, 28)).flatten() / 255.0

pred = model.predict([test_feat])[0]
conf = model.predict_proba([test_feat])[0][pred] * 100

gender_text = "Female (เพศหญิง)" if pred == 1 else "Male (เพศชาย)"
print(f'รูปภาพทดสอบ: {sample_img_path}')
print(f'1. เพศสภาพ (Sex): {gender_text} (ความมั่นใจ: {conf:.1f}%)')